# Stage 2: Safety Score Prediction (200k)

This notebook predicts the perceived cycling safety class of a street-level image from visual features extracted by a computer vision pipeline. The target variable is derived from TrueSkill rankings produced in Stage 1.

**Pipeline:**
1. Install dependencies
2. Imports
3. Configuration
4. Load datasets
5. Composite feature construction
6. Feature cleaning (9 steps)
7. Feature family definitions
8. Model definitions
9. Shared functions
10. Tree hyperparameter search
11. Model runner functions
12. Plotting functions
13. Main CV loop (location-grouped)
14. Results
15. Spatial block cross-validation (London Plan sub-regions)
16. Feature importance (SHAP)

**Datasets:**
- Visual features: segmentation coverage, centroid, perimeter, grid, object detection, image statistics
- GPT TrueSkill ranking: based on 200k synthetic votes
- Human TrueSkill ranking: based on 19,729 filtered crowdsourced votes
- Borough assignment: London Plan five sub-regions (point_borough_assignment.csv)

**Target:** 3-class safety label (Low / Medium / High) from quantile binning of TrueSkill mu

**Models:** Deeper MLP (256->128->64->32), XGBoost, LightGBM, CatBoost

**Evaluation:**
- Primary: 3 independent partitions x 5-fold location-level GroupKFold = 15 runs per model
- Secondary: 5-fold spatial block CV using London Plan sub-regions (best model per dataset only)

In [ ]:
## 1. Install dependencies
import sys, subprocess
print('Installing dependencies...')
for pkg in ['xgboost', 'lightgbm', 'catboost', 'shap']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
print('Done.')

In [ ]:
## 2. Imports
print('Loading libraries...')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
import re
import sklearn
import sklearn.base
import shap
warnings.filterwarnings('ignore')
from scipy.stats import pearsonr

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit, train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, make_scorer
from sklearn.utils.class_weight import compute_class_weight

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)
print('Imported.')

In [ ]:
## 3. Configuration
# Edit KAGGLE_INPUT before running.

KAGGLE_INPUT     = '/kaggle/input/datasets/lsestudent/insumos2'
FEATURES_PATH    = f'{KAGGLE_INPUT}/processed_features_merged_02_06_26.csv'
GPT_RANK_PATH    = f'{KAGGLE_INPUT}/gpt_trueskill_rankings_200k_20_06_26.csv'
HUMAN_RANK_PATH  = f'{KAGGLE_INPUT}/human_trueskill_rankings_final_20_06_26.csv'
BOROUGH_PATH     = f'{KAGGLE_INPUT}/point_borough_assignment.csv'

SIGMA_WEIGHTING       = '1/sigma^2'
PEARSON_THRESHOLD_GPT   = 0.03
PEARSON_THRESHOLD_HUMAN = 0.03

NUM_BINS              = 3
BINNING_STRATEGY      = 'qcut'
BIN_LABELS            = ['Low', 'Medium', 'High']
N_SPLITS              = 5
N_PARTITIONS          = 3
VAL_SIZE_WITHIN_TRAIN = 0.15

EPOCHS        = 150
BATCH_SIZE    = 64
PATIENCE      = 20
N_ITER_SEARCH = 12
CV_SEARCH     = 3
IMAGE_PIXELS  = 640 * 640

FONT_FAMILY    = 'DejaVu Sans'
plt.rcParams['font.family'] = FONT_FAMILY
DATASET_COLORS = {'GPT': '#2171B5', 'Human': '#1B7837'}

print('Configuration loaded.')
print('=' * 55)
print(f'  Features path    : {FEATURES_PATH}')
print(f'  GPT ranking      : {GPT_RANK_PATH}')
print(f'  Human ranking    : {HUMAN_RANK_PATH}')
print(f'  Borough path     : {BOROUGH_PATH}')
print(f'  Bins             : {NUM_BINS} ({", ".join(BIN_LABELS)})')
print(f'  Binning strategy : {BINNING_STRATEGY}')
print(f'  Pearson GPT      : |r| > {PEARSON_THRESHOLD_GPT}')
print(f'  Pearson Human    : |r| > {PEARSON_THRESHOLD_HUMAN}')
print(f'  Sigma weighting  : {SIGMA_WEIGHTING}')
print(f'  CV               : {N_PARTITIONS} x {N_SPLITS} = {N_PARTITIONS*N_SPLITS} evals per model')
print(f'  Random baseline  : {1/NUM_BINS*100:.2f}%')

In [ ]:
## 4. Load datasets
print('Loading datasets...')
import os
raw_features_df = pd.read_csv(FEATURES_PATH)
gpt_df          = pd.read_csv(GPT_RANK_PATH)
human_df        = pd.read_csv(HUMAN_RANK_PATH)
borough_df      = pd.read_csv(BOROUGH_PATH)

gpt_df['image_name']   = gpt_df['image_name'].apply(lambda x: os.path.basename(str(x).strip()))
human_df['image_name'] = human_df['image_name'].apply(lambda x: os.path.basename(str(x).strip()))

print('=' * 55)
print(f'  Features : {raw_features_df.shape[0]:,} rows x {raw_features_df.shape[1]:,} columns')
print(f'  GPT      : {gpt_df.shape[0]:,} rows | columns: {list(gpt_df.columns)}')
print(f'  Human    : {human_df.shape[0]:,} rows | columns: {list(human_df.columns)}')
print(f'  Borough  : {borough_df.shape[0]:,} rows | sub-regions: {borough_df["subregion"].nunique()}')
print(f'  GPT   sigma range : [{gpt_df["gpt_sigma"].min():.4f}, {gpt_df["gpt_sigma"].max():.4f}]')
print(f'  Human sigma range : [{human_df["human_sigma"].min():.4f}, {human_df["human_sigma"].max():.4f}]')
print()
print('Sub-region distribution in borough file:')
print(borough_df['subregion'].value_counts().sort_index().to_string())

In [ ]:
## 5. Composite feature construction
# (unchanged from original — see original notebook for comments)

def safe_sum(df_, cols):
    present = [c for c in cols if c in df_.columns]
    missing = [c for c in cols if c not in df_.columns]
    if missing:
        print(f'  WARNING: missing columns skipped: {missing}')
    return df_[present].fillna(0).sum(axis=1) if present else pd.Series(0, index=df_.index)

df_with_composites = raw_features_df.copy()

df_with_composites['comp_cycling_infra']       = safe_sum(df_with_composites, ['seg_Bike Lane', 'seg_Bike Rack', 'seg_Bicycle', 'seg_Bicyclist', 'seg_Other Rider', 'seg_Wheeled Slow'])
df_with_composites['comp_motor_load']          = safe_sum(df_with_composites, ['seg_Car', 'seg_Truck', 'seg_Bus', 'seg_Motorcycle', 'seg_Other Vehicle'])
df_with_composites['comp_separation']          = safe_sum(df_with_composites, ['seg_Barrier', 'seg_Guard Rail', 'seg_Fence', 'seg_Wall', 'seg_Tunnel', 'seg_Bridge'])
df_with_composites['comp_natural_openness']    = safe_sum(df_with_composites, ['seg_Vegetation', 'seg_Terrain', 'seg_Mountain', 'seg_Water'])
df_with_composites['comp_commercial_density']  = safe_sum(df_with_composites, ['seg_Billboard', 'seg_Street Light', 'seg_Traffic Sign (Front)', 'seg_CCTV Camera'])
df_with_composites['comp_built_density']       = safe_sum(df_with_composites, ['seg_Building', 'seg_Wall', 'seg_Bridge'])

for suffix in ['cycling_infra', 'motor_load', 'separation', 'natural_openness', 'commercial_density', 'built_density']:
    df_with_composites[f'comp_has_{suffix}'] = (df_with_composites[f'comp_{suffix}'] > 0).astype(int)

df_with_composites['comp_nature_vs_built']  = df_with_composites['comp_natural_openness'] / (df_with_composites['comp_built_density'] + 0.001)
df_with_composites['comp_terrain_vs_motor'] = safe_sum(df_with_composites, ['seg_Terrain', 'seg_Vegetation']) / (df_with_composites['comp_motor_load'] + 0.001)

comp_cols = [c for c in df_with_composites.columns if c.startswith('comp_')]
print(f'Composite features added : {len(comp_cols)}')
print(f'Shape before cleaning    : {df_with_composites.shape[0]:,} rows x {df_with_composites.shape[1]:,} columns')

In [ ]:
## 6. Feature cleaning (9 steps)
# (unchanged from original)

df            = df_with_composites.copy()
initial_shape = df.shape
_grid_re      = re.compile(r'_\d+_\d+$')
_all_raw      = [c for c in df.columns if c != 'image_name']
spatial_cols  = (
    [c for c in _all_raw if c.startswith('seg_') and c.endswith('_cx')] +
    [c for c in _all_raw if c.startswith('seg_') and c.endswith('_cy')] +
    [c for c in _all_raw if c.startswith('seg_') and c.endswith('_perimeter')]
)
area_cols   = [c for c in _all_raw if c.startswith('obj_') and '_area_' in c]
non_spatial = [c for c in _all_raw if c not in spatial_cols]

unnamed = [c for c in df.columns if str(c).startswith('Unnamed') or not str(c).strip()]
df      = df.drop(columns=unnamed) if unnamed else df
print(f'Step 1 — unnamed columns removed              : {len(unnamed)}')

df['point_id'] = df['image_name'].str.extract(r'^(\d+)_')
n_locs = df['point_id'].nunique()
print(f'Step 2 — point_id extracted                   : {n_locs:,} unique locations')

non_spatial_present = [c for c in non_spatial if c in df.columns]
nan_non_spatial = int(df[non_spatial_present].isna().sum().sum())
nan_spatial     = int(df[spatial_cols].isna().sum().sum()) if spatial_cols else 0
df[non_spatial_present] = df[non_spatial_present].fillna(0)
print(f'Step 3 — coverage/grid NaNs filled with 0     : {nan_non_spatial:,}')
print(f'         centroid/perimeter NaNs kept as NaN  : {nan_spatial:,}')

check_cols    = [c for c in non_spatial_present if not c.startswith('comp_')]
all_zero_mask = (df[check_cols] == 0).all(axis=1)
invalid_imgs  = df.loc[all_zero_mask, 'image_name'].tolist()
df            = df[~all_zero_mask].copy()
print(f'Step 4 — invalid all-zero rows removed        : {len(invalid_imgs)}')

dup_check = [c for c in non_spatial_present if c in df.columns and not c.startswith('comp_')]
dup_mask  = df.duplicated(subset=dup_check, keep='first')
df        = df[~dup_mask].copy()
print(f'Step 5 — exact duplicate rows removed         : {dup_mask.sum()}')

sky_points = ['2285', '2790']
sky_mask   = df['point_id'].isin(sky_points)
df         = df[~sky_mask].copy()
print(f'Step 6 — sky-only atypical images removed     : {sky_mask.sum()}')

area_present = [c for c in area_cols if c in df.columns]
already_norm = (df[area_present].max().max() <= 1.0) if area_present else False
if area_present and not already_norm:
    df[area_present] = df[area_present] / IMAGE_PIXELS
    print(f'Step 7 — object area normalised by {IMAGE_PIXELS:,} pixels : {len(area_present)} columns')
else:
    print(f'Step 7 — object area already normalised       : skipped')

feat_check  = [c for c in df.columns if c not in ('image_name', 'point_id')]
zv_cols     = [c for c in feat_check if df[c].nunique() <= 1]
df          = df.drop(columns=zv_cols)
print(f'Step 8 — zero-variance columns removed        : {len(zv_cols)}')

feat_check2 = [c for c in df.columns if c not in ('image_name', 'point_id')]
nz_frac     = (df[feat_check2] != 0).mean()
near_zero   = nz_frac[nz_frac < 0.001].index.tolist()
df          = df.drop(columns=near_zero)
print(f'Step 9 — near-zero columns removed            : {len(near_zero)}')

features_df = df.copy()
final_feats = [c for c in features_df.columns if c not in ('image_name', 'point_id')]
print(f'\nFinal feature matrix: {len(features_df):,} rows x {len(final_feats)} features')

In [ ]:
## 7. Feature family definitions

def get_family(feat):
    if feat.startswith('comp_'):                                    return 'Composite'
    if feat.startswith('seg_') and feat.endswith('_cx'):           return 'Seg centroid X'
    if feat.startswith('seg_') and feat.endswith('_cy'):           return 'Seg centroid Y'
    if feat.startswith('seg_') and feat.endswith('_perimeter'):    return 'Seg perimeter'
    if feat.startswith('seg_') and re.search(r'_\d+_\d+$', feat): return 'Seg grid'
    if feat.startswith('seg_'):                                     return 'Seg coverage'
    if feat.startswith('obj_') and '_prob_'  in feat:              return 'Obj probability'
    if feat.startswith('obj_') and '_count_' in feat:              return 'Obj count'
    if feat.startswith('obj_') and '_area_'  in feat:              return 'Obj area'
    return 'Image statistics'

ALL_FAMILIES = [
    'Composite', 'Seg coverage', 'Seg centroid X', 'Seg centroid Y',
    'Seg perimeter', 'Seg grid', 'Obj probability', 'Obj count',
    'Obj area', 'Image statistics'
]

family_counts = {fam: 0 for fam in ALL_FAMILIES}
for f in final_feats:
    fam = get_family(f)
    family_counts[fam] = family_counts.get(fam, 0) + 1

print('Feature family counts:')
for fam, cnt in family_counts.items():
    print(f'  {fam:<25}: {cnt}')
print(f'  Total: {sum(family_counts.values())}')

In [ ]:
## 8. Model definitions

def create_deeper_mlp(input_dim, num_classes):
    model = Sequential([
        Dense(256, input_shape=(input_dim,)),
        BatchNormalization(), Activation('relu'), Dropout(0.3),
        Dense(128),
        BatchNormalization(), Activation('relu'), Dropout(0.3),
        Dense(64),
        BatchNormalization(), Activation('relu'), Dropout(0.3),
        Dense(32),
        Activation('relu'), Dropout(0.2),
        Dense(num_classes, activation='softmax')
    ], name='Deeper_MLP')
    model.compile(
        optimizer=Adam(0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

MODEL_ORDER = ['Deeper_MLP', 'XGBoost', 'LightGBM', 'CatBoost']
print('Models defined.')
print('  Deeper_MLP : 256->128->64->32->output')
print('  XGBoost    : gradient boosting, level-wise trees')
print('  LightGBM   : gradient boosting, leaf-wise trees')
print('  CatBoost   : gradient boosting, symmetric oblivious trees')

In [ ]:
## 9. Shared functions

def bin_mu(series):
    return pd.qcut(series, q=NUM_BINS, labels=False, duplicates='drop')

def select_features_pearson(features_df_in, merged_df, mu_col, threshold):
    all_feats = [c for c in features_df_in.columns
                 if c not in ('image_name', 'point_id') and c in merged_df.columns]
    mu_values = merged_df[mu_col].values
    corr_vals = {}
    for feat in all_feats:
        x = merged_df[feat].replace([np.inf, -np.inf], np.nan).fillna(0).values
        corr_vals[feat] = 0.0 if x.std() == 0 else float(pearsonr(x, mu_values)[0])
    corr_df = (
        pd.DataFrame({'feature': list(corr_vals.keys()), 'pearson_r': list(corr_vals.values())})
          .assign(abs_r=lambda d: d['pearson_r'].abs())
          .sort_values('abs_r', ascending=False)
          .reset_index(drop=True)
    )
    selected = corr_df[corr_df['abs_r'] > threshold]['feature'].tolist()
    print(f'  Pearson threshold : |r| > {threshold}')
    print(f'  Features kept     : {len(selected)} / {len(all_feats)}')
    print(f'  Top 10 by |r|:')
    for _, row in corr_df.head(10).iterrows():
        print(f'    {row["feature"]:<45s} r={row["pearson_r"]:+.4f}')
    return selected, corr_df

def prep_data(features_df_in, votes_df, mu_col, sigma_col, threshold):
    merged = votes_df.merge(features_df_in, on='image_name', how='inner')
    merged['mu_bin'] = bin_mu(merged[mu_col])
    print('  Computing Pearson correlations with raw mu...')
    selected, corr_df = select_features_pearson(features_df_in, merged, mu_col, threshold)
    zv = [f for f in selected if merged[f].nunique() <= 1]
    if zv:
        selected = [f for f in selected if f not in zv]
        print(f'  Zero-variance post-merge dropped: {len(zv)}')
    return merged, selected, corr_df

def get_sample_weights(sigma_values, y_values, class_weights):
    if SIGMA_WEIGHTING == 'none':
        sw = np.ones(len(sigma_values))
    elif SIGMA_WEIGHTING == '1/sigma':
        sw = 1.0 / np.maximum(sigma_values, 1e-6)
    elif SIGMA_WEIGHTING == '1/sigma^2':
        sw = 1.0 / np.maximum(sigma_values ** 2, 1e-12)
    else:
        raise ValueError(f'Unknown SIGMA_WEIGHTING: {SIGMA_WEIGHTING}')
    sw       = sw / sw.mean()
    cw       = np.array([class_weights[yi] for yi in y_values])
    combined = sw * cw
    return combined / combined.mean()

def preprocess_for_mlp(X_tr, X_val, X_te):
    imp    = SimpleImputer(strategy='constant', fill_value=0.5)
    scaler = StandardScaler()
    X_tr_i  = imp.fit_transform(X_tr)
    X_val_i = imp.transform(X_val)
    X_te_i  = imp.transform(X_te)
    X_tr_s  = scaler.fit_transform(X_tr_i)
    X_val_s = scaler.transform(X_val_i)
    X_te_s  = scaler.transform(X_te_i)
    return X_tr_s, X_val_s, X_te_s

print('Shared functions defined.')

In [ ]:
## 10. Tree hyperparameter search

def tune_trees(X, y, sigma, groups, label):
    print(f'  Tuning tree models for {label}...')
    gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
    tr_idx, _ = next(gss.split(X, y, groups=groups))
    X_s   = X.iloc[tr_idx]
    y_s   = y.iloc[tr_idx]
    sig_s = sigma.iloc[tr_idx]
    cw_arr = compute_class_weight('balanced', classes=np.unique(y_s), y=y_s)
    cw     = {i: w for i, w in enumerate(cw_arr)}
    sw     = get_sample_weights(sig_s.values, y_s.values, cw)
    scorer = make_scorer(balanced_accuracy_score)

    xgb_search = RandomizedSearchCV(
        XGBClassifier(subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss', random_state=SEED, verbosity=0),
        {'n_estimators': [200, 300, 400], 'max_depth': [4, 6, 8], 'learning_rate': [0.05, 0.10]},
        n_iter=N_ITER_SEARCH, scoring=scorer, cv=CV_SEARCH, random_state=SEED, n_jobs=-1, refit=False
    )
    xgb_search.fit(X_s.fillna(0), y_s, sample_weight=sw)
    best_xgb = xgb_search.best_params_
    print(f'  XGBoost  best: {best_xgb}  BA={xgb_search.best_score_:.4f}')

    lgb_search = RandomizedSearchCV(
        LGBMClassifier(subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbosity=-1),
        {'n_estimators': [200, 300, 400], 'max_depth': [4, 6, 8, -1], 'learning_rate': [0.05, 0.10], 'num_leaves': [31, 63, 127]},
        n_iter=N_ITER_SEARCH, scoring=scorer, cv=CV_SEARCH, random_state=SEED, n_jobs=-1, refit=False
    )
    lgb_search.fit(X_s.fillna(0), y_s, sample_weight=sw)
    best_lgb = lgb_search.best_params_
    print(f'  LightGBM best: {best_lgb}  BA={lgb_search.best_score_:.4f}')

    cat_search = RandomizedSearchCV(
        CatBoostClassifier(random_seed=SEED, verbose=0),
        {'iterations': [200, 300, 400], 'depth': [4, 6, 8], 'learning_rate': [0.05, 0.10]},
        n_iter=N_ITER_SEARCH, scoring=scorer, cv=CV_SEARCH, random_state=SEED, n_jobs=-1, refit=False
    )
    cat_search.fit(X_s.fillna(0), y_s, sample_weight=sw)
    best_cat = cat_search.best_params_
    print(f'  CatBoost best: {best_cat}  BA={cat_search.best_score_:.4f}')

    return {'xgb': best_xgb, 'lgb': best_lgb, 'cat': best_cat}

print('=' * 70)
print('HYPERPARAMETER SEARCH')
print('=' * 70)
print(f'20% location-level hold-out | {N_ITER_SEARCH} iters each | {CV_SEARCH}-fold | sigma weights')
print()

feat_cols_all = [c for c in features_df.columns if c not in ('image_name', 'point_id')]
tree_params   = {}

for ds_label, df_, mu_col, sigma_col, thr in [
    ('GPT',   gpt_df,   'gpt_mu',   'gpt_sigma',   PEARSON_THRESHOLD_GPT),
    ('Human', human_df, 'human_mu', 'human_sigma', PEARSON_THRESHOLD_HUMAN),
]:
    merged_s           = df_.merge(features_df, on='image_name', how='inner')
    merged_s['mu_bin'] = bin_mu(merged_s[mu_col])
    X_s = merged_s[feat_cols_all]
    y_s = merged_s['mu_bin']
    s_s = merged_s[sigma_col]
    g_s = merged_s['point_id']
    tree_params[ds_label] = tune_trees(X_s, y_s, s_s, g_s, ds_label)
    print()

print('Search complete.')

In [ ]:
## 12. Plotting functions

def print_results_table(results_all):
    header = f'  {"Model":<20} {"Type":<8}  {"GPT BA":>24}  {"Human BA":>24}'
    print(header)
    print('-' * 78)
    for mname in MODEL_ORDER:
        gpt_r   = next((r for r in results_all['GPT']   if r['model'] == mname), None)
        hum_r   = next((r for r in results_all['Human'] if r['model'] == mname), None)
        mtype   = gpt_r['type'] if gpt_r else ''
        gpt_str = f'{gpt_r["bal_acc_mean"]:.4f} +/- {gpt_r["bal_acc_std"]:.4f}' if gpt_r else 'N/A'
        hum_str = f'{hum_r["bal_acc_mean"]:.4f} +/- {hum_r["bal_acc_std"]:.4f}' if hum_r else 'N/A'
        print(f'  {mname:<20} {mtype:<8}  {gpt_str:>24}  {hum_str:>24}')
    print('-' * 78)
    print(f'  Random baseline{"":>30}  {1/NUM_BINS:.4f}{"":>20}  {1/NUM_BINS:.4f}')


def plot_results_bar(results_all):
    models    = MODEL_ORDER
    gpt_means = [next(r['bal_acc_mean'] for r in results_all['GPT']   if r['model'] == m) for m in models]
    gpt_stds  = [next(r['bal_acc_std']  for r in results_all['GPT']   if r['model'] == m) for m in models]
    hum_means = [next(r['bal_acc_mean'] for r in results_all['Human'] if r['model'] == m) for m in models]
    hum_stds  = [next(r['bal_acc_std']  for r in results_all['Human'] if r['model'] == m) for m in models]
    baseline  = 1 / NUM_BINS
    x     = np.arange(len(models))
    width = 0.32
    fig, ax = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    bars_gpt = ax.bar(x - width/2, gpt_means, width, yerr=gpt_stds, capsize=4,
                      color=DATASET_COLORS['GPT'], alpha=0.85, edgecolor='white', label='GPT-5.2 (200k votes)')
    bars_hum = ax.bar(x + width/2, hum_means, width, yerr=hum_stds, capsize=4,
                      color=DATASET_COLORS['Human'], alpha=0.85, edgecolor='white', label='Human (19,729 filtered votes)')
    for bar, mean, std in zip(bars_gpt, gpt_means, gpt_stds):
        ax.text(bar.get_x()+bar.get_width()/2, mean+std+0.007, f'{mean*100:.1f}%',
                ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#111111')
    for bar, mean, std in zip(bars_hum, hum_means, hum_stds):
        ax.text(bar.get_x()+bar.get_width()/2, mean+std+0.007, f'{mean*100:.1f}%',
                ha='center', va='bottom', fontsize=8.5, fontweight='bold', color='#111111')
    ax.axhline(baseline, color='#c0392b', linestyle='--', linewidth=1.6, alpha=0.9,
               label=f'Random baseline ({baseline*100:.1f}%)')
    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', ' ') for m in models], fontsize=11)
    ax.set_ylabel('Balanced accuracy', fontsize=11)
    ax.set_ylim(0, 0.74)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v*100:.0f}%'))
    ax.set_title('Safety score prediction: balanced accuracy by model and dataset\n'
                 f'3-class qcut  |  1/\u03c3\u00b2 sample weighting  |  Location-level GroupKFold  |  3\u00d75 CV',
                 fontsize=11, fontweight='bold', pad=10)
    ax.legend(fontsize=10, frameon=True, framealpha=0.95, loc='lower right', edgecolor='#cccccc')
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines['left'].set_color('#cccccc')
    ax.spines['bottom'].set_color('#cccccc')
    ax.grid(axis='y', alpha=0.25, linestyle='--', color='#aaaaaa')
    ax.tick_params(labelsize=10)
    plt.tight_layout()
    plt.show()


def plot_confusion_and_recall(results_all):
    for ds_label in ['GPT', 'Human']:
        best = max(results_all[ds_label], key=lambda r: r['bal_acc_mean'])
        cm   = best['confusion_matrix']
        print(f'  Best {ds_label} model : {best["model"]} (BA={best["bal_acc_mean"]:.4f})')
        row_sums = cm.sum(axis=1, keepdims=True)
        cm_n     = np.where(row_sums > 0, cm / row_sums, 0.0)
        print(f'  Bin  {"Label":<12} {"Recall":>8} {"Precision":>10} {"F1":>8} {"Support":>10}')
        print('  ' + '-' * 58)
        for i, lbl in enumerate(BIN_LABELS):
            tp  = cm[i, i]
            rec = cm_n[i, i]
            pre = tp / cm[:, i].sum() if cm[:, i].sum() > 0 else 0
            f1  = 2*pre*rec/(pre+rec) if (pre+rec) > 0 else 0
            sup = int(cm[i].sum())
            print(f'  {i}    {lbl:<12} {rec:>8.3f} {pre:>10.3f} {f1:>8.3f} {sup:>10,}')
        print('  ' + '-' * 58)
        mean_rec = cm_n.diagonal().mean()
        print(f'  Mean{"":>17} {mean_rec:>8.3f}')
        print()


def plot_feature_family_importance(importances_ds, ds_label):
    feat_names, imp_vals = importances_ds['LightGBM'] if ds_label == 'GPT' else importances_ds['CatBoost']
    family_imp = {}
    for feat, imp in zip(feat_names, imp_vals):
        fam = get_family(feat)
        family_imp[fam] = family_imp.get(fam, 0.0) + imp
    sorted_fams = sorted(family_imp.items(), key=lambda x: x[1], reverse=True)
    fam_names = [f[0] for f in sorted_fams]
    fam_vals  = [f[1] for f in sorted_fams]
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.barh(range(len(fam_names)), fam_vals[::-1],
            color=DATASET_COLORS[ds_label], alpha=0.85, edgecolor='white')
    ax.set_yticks(range(len(fam_names)))
    ax.set_yticklabels(fam_names[::-1], fontsize=9)
    ax.set_xlabel('Sum of feature importances within family', fontsize=9)
    ax.set_title(f'Feature family importance — {ds_label}', fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=0.2, linestyle='--')
    plt.tight_layout()
    plt.show()


print('Plotting functions defined.')

In [ ]:
## 11. Model runner functions

def run_all_models(X, y, sigma, groups, label, tree_params_ds):
    results     = []
    importances = {}
    feat_names  = list(X.columns)
    num_classes = NUM_BINS

    print(f'  Deeper_MLP ({N_PARTITIONS}x{N_SPLITS} folds)...')
    all_scores, all_cms, all_wmags = [], [], []
    for part in range(N_PARTITIONS):
        sgkf       = StratifiedGroupKFold(n_splits=N_SPLITS)
        split_iter = list(sgkf.split(X, y, groups=groups))
        rng        = np.random.RandomState(SEED + part * 100)
        rng.shuffle(split_iter)
        for fold, (tr_idx, te_idx) in enumerate(split_iter):
            X_trf  = X.iloc[tr_idx];     y_trf  = y.iloc[tr_idx]
            s_trf  = sigma.iloc[tr_idx]; X_te   = X.iloc[te_idx]
            y_te   = y.iloc[te_idx]
            tr_i, val_i = train_test_split(
                np.arange(len(X_trf)), test_size=VAL_SIZE_WITHIN_TRAIN,
                stratify=y_trf.values, random_state=SEED + part * 100 + fold
            )
            X_tr  = X_trf.iloc[tr_i];  y_tr  = y_trf.iloc[tr_i];  s_tr  = s_trf.iloc[tr_i]
            X_val = X_trf.iloc[val_i]; y_val = y_trf.iloc[val_i]; s_val = s_trf.iloc[val_i]
            X_tr_s, X_val_s, X_te_s = preprocess_for_mlp(X_tr, X_val, X_te)
            cw_arr = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
            cw     = {i: w for i, w in enumerate(cw_arr)}
            sw_tr  = get_sample_weights(s_tr.values,  y_tr.values,  cw)
            sw_val = get_sample_weights(s_val.values, y_val.values, cw)
            model  = create_deeper_mlp(X_tr_s.shape[1], num_classes)
            model.fit(
                X_tr_s, y_tr,
                validation_data=(X_val_s, y_val, sw_val),
                epochs=EPOCHS, batch_size=BATCH_SIZE, sample_weight=sw_tr,
                callbacks=[
                    EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True, verbose=0),
                    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=0)
                ], verbose=0
            )
            y_pred = np.argmax(model.predict(X_te_s, verbose=0), axis=1)
            all_scores.append(balanced_accuracy_score(y_te, y_pred))
            all_cms.append(confusion_matrix(y_te, y_pred, labels=list(range(num_classes))))
            all_wmags.append(np.abs(model.layers[0].get_weights()[0]).mean(axis=1))
    mean_ba = np.mean(all_scores); std_ba = np.std(all_scores)
    results.append({'model': 'Deeper_MLP', 'type': 'MLP',
                    'bal_acc_mean': mean_ba, 'bal_acc_std': std_ba,
                    'bal_acc_min': np.min(all_scores), 'bal_acc_max': np.max(all_scores),
                    'confusion_matrix': np.mean(all_cms, axis=0)})
    importances['Deeper_MLP'] = (feat_names, np.mean(all_wmags, axis=0))
    print(f'    >> {mean_ba:.4f} +/- {std_ba:.4f}  [{np.min(all_scores):.4f}, {np.max(all_scores):.4f}]')

    tree_cfgs = [
        ('XGBoost',  XGBClassifier(**tree_params_ds['xgb'], subsample=0.8, colsample_bytree=0.8, eval_metric='mlogloss', random_state=SEED, verbosity=0)),
        ('LightGBM', LGBMClassifier(**tree_params_ds['lgb'], subsample=0.8, colsample_bytree=0.8, random_state=SEED, verbosity=-1)),
        ('CatBoost', CatBoostClassifier(**tree_params_ds['cat'], random_seed=SEED, verbose=0)),
    ]
    for model_name, clf_template in tree_cfgs:
        print(f'  {model_name} ({N_PARTITIONS}x{N_SPLITS} folds)...')
        all_scores_t, all_cms_t, all_imp_t = [], [], []
        for part in range(N_PARTITIONS):
            sgkf       = StratifiedGroupKFold(n_splits=N_SPLITS)
            split_iter = list(sgkf.split(X, y, groups=groups))
            rng        = np.random.RandomState(SEED + part * 100)
            rng.shuffle(split_iter)
            for tr_idx, te_idx in split_iter:
                X_tr = X.iloc[tr_idx]; y_tr = y.iloc[tr_idx]; s_tr = sigma.iloc[tr_idx]
                X_te = X.iloc[te_idx]; y_te = y.iloc[te_idx]
                cw_arr = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
                cw     = {i: w for i, w in enumerate(cw_arr)}
                sw_tr  = get_sample_weights(s_tr.values, y_tr.values, cw)
                clf    = sklearn.base.clone(clf_template)
                clf.fit(X_tr, y_tr, sample_weight=sw_tr)
                y_pred = clf.predict(X_te)
                all_scores_t.append(balanced_accuracy_score(y_te, y_pred))
                all_cms_t.append(confusion_matrix(y_te, y_pred, labels=list(range(num_classes))))
                all_imp_t.append(clf.feature_importances_)
        mean_ba = np.mean(all_scores_t); std_ba = np.std(all_scores_t)
        key     = {'XGBoost': 'xgb', 'LightGBM': 'lgb', 'CatBoost': 'cat'}[model_name]
        results.append({'model': model_name, 'type': 'Tree',
                        'bal_acc_mean': mean_ba, 'bal_acc_std': std_ba,
                        'bal_acc_min': np.min(all_scores_t), 'bal_acc_max': np.max(all_scores_t),
                        'confusion_matrix': np.mean(all_cms_t, axis=0),
                        'best_params': tree_params_ds[key]})
        importances[model_name] = (feat_names, np.mean(all_imp_t, axis=0))
        print(f'    >> {mean_ba:.4f} +/- {std_ba:.4f}  [{np.min(all_scores_t):.4f}, {np.max(all_scores_t):.4f}]')

    return results, importances


def run_single_model(X, y, sigma, groups, model_name, model_obj):
    all_scores = []
    for part in range(N_PARTITIONS):
        sgkf       = StratifiedGroupKFold(n_splits=N_SPLITS)
        split_iter = list(sgkf.split(X, y, groups=groups))
        rng        = np.random.RandomState(SEED + part * 100)
        rng.shuffle(split_iter)
        for fold, (tr_idx, te_idx) in enumerate(split_iter):
            X_trf = X.iloc[tr_idx]; y_trf = y.iloc[tr_idx]; s_trf = sigma.iloc[tr_idx]
            X_te  = X.iloc[te_idx]; y_te  = y.iloc[te_idx]
            cw_arr = compute_class_weight('balanced', classes=np.unique(y_trf), y=y_trf)
            cw     = {i: w for i, w in enumerate(cw_arr)}
            sw_tr  = get_sample_weights(s_trf.values, y_trf.values, cw)
            clf    = sklearn.base.clone(model_obj)
            clf.fit(X_trf, y_trf, sample_weight=sw_tr)
            y_pred = clf.predict(X_te)
            all_scores.append(balanced_accuracy_score(y_te, y_pred))
    return np.mean(all_scores), np.std(all_scores)


print('Model runner functions defined.')

In [ ]:
## 13. Main CV loop (location-grouped)

print('=' * 70)
print(f'DATA PREPARATION — {NUM_BINS} BINS ({BINNING_STRATEGY.upper()})')
print('=' * 70)

merged_all    = {}
feats_all     = {}
sigma_cols    = {'GPT': 'gpt_sigma', 'Human': 'human_sigma'}
results_all   = {}
importances_all = {}

for ds_label, df_, mu_col, sigma_col, thr in [
    ('GPT',   gpt_df,   'gpt_mu',   'gpt_sigma',   PEARSON_THRESHOLD_GPT),
    ('Human', human_df, 'human_mu', 'human_sigma', PEARSON_THRESHOLD_HUMAN),
]:
    print(f'\n{ds_label}:')
    merged, selected, _ = prep_data(features_df, df_, mu_col, sigma_col, thr)

        # Merge borough/subregion assignment
    merged['point_id'] = merged['point_id'].astype(str)
    borough_df['point_id'] = borough_df['point_id'].astype(str)
    merged = merged.merge(borough_df[['point_id', 'borough_name', 'subregion']],
                          on='point_id', how='left')

    bin_dist = merged['mu_bin'].value_counts().sort_index()
    print(f'  Merged rows : {len(merged):,} | Features selected: {len(selected)}')
    print(f'  Locations   : {merged["point_id"].nunique():,}')
    print(f'  Bin distribution: ' +
          ' | '.join([f'Bin {k}={v} ({v/len(merged)*100:.0f}%)' for k, v in bin_dist.items()]))
    merged_all[ds_label] = merged
    feats_all[ds_label]  = selected

for ds_label in ['GPT', 'Human']:
    print()
    print('=' * 70)
    print(f'EXPERIMENTS — {ds_label} ({NUM_BINS} BINS)')
    print('=' * 70)
    m      = merged_all[ds_label]
    feats  = feats_all[ds_label]
    X      = m[feats].reset_index(drop=True)
    y      = m['mu_bin'].reset_index(drop=True)
    sigma  = m[sigma_cols[ds_label]].reset_index(drop=True)
    groups = m['point_id'].reset_index(drop=True)
    print(f'  Samples  : {len(X):,} | Features: {X.shape[1]}')
    print(f'  Locations: {groups.nunique():,} (GroupKFold — no location leakage)')
    print(f'  Random baseline: {1/NUM_BINS*100:.1f}%')
    print(f'  CV: {N_PARTITIONS} x {N_SPLITS} = {N_PARTITIONS*N_SPLITS} evaluations per model')
    print()
    res, imp = run_all_models(X, y, sigma, groups, ds_label, tree_params[ds_label])
    results_all[ds_label]     = res
    importances_all[ds_label] = imp
    print(f'\n{ds_label} experiments complete.')

lookups_all = {
    ds: {r['model']: r for r in res}
    for ds, res in results_all.items()
}
print('\nAll models complete.')

In [ ]:
## 14. Results

print('\n======================================================================')
print(f'RESULTS SUMMARY — {NUM_BINS} BINS ({BINNING_STRATEGY.upper()}, {SIGMA_WEIGHTING} WEIGHTING)')
print('======================================================================')
print_results_table(results_all)
print()
plot_results_bar(results_all)

print('======================================================================')
print('CONFUSION MATRICES + RECALL — BEST MODEL PER DATASET')
print('======================================================================')
plot_confusion_and_recall(results_all)

print('======================================================================')
print('FEATURE FAMILY IMPORTANCE (native importances)')
print('======================================================================')
for ds_label in ['GPT', 'Human']:
    print(f'\n--- {ds_label} ---')
    plot_feature_family_importance(importances_all[ds_label], ds_label)

In [ ]:
## 14b. Confusion matrices for the best model per dataset (real, from CV folds)
def plot_confusion_matrices_best_models():
    BLUE   = '#2171B5'
    ORANGE = '#E8743B'
    BIN_LABELS = ['Low', 'Medium', 'High']

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.patch.set_facecolor('white')
    fig.suptitle(
        'Confusion matrices — best model per dataset\n'
        'Row-normalised: diagonal = recall per class',
        fontsize=12, fontweight='bold', y=1.02
    )

    for ax, ds_label, color in zip(axes, ['GPT', 'Human'], [BLUE, ORANGE]):
        best_result = max(results_all[ds_label], key=lambda r: r['bal_acc_mean'])
        best_name   = best_result['model']
        best_ba     = best_result['bal_acc_mean']
        cm          = best_result['confusion_matrix']  # mean over 15 folds, may be non-integer

        row_sums = cm.sum(axis=1, keepdims=True)
        cm_n     = np.where(row_sums > 0, cm / row_sums, 0.0)
        recall   = np.diag(cm_n)
        support  = row_sums.flatten()

        ax.set_facecolor('white')
        im = ax.imshow(cm_n, cmap='Blues', vmin=0, vmax=1, aspect='equal')

        for i in range(3):
            for j in range(3):
                val = cm_n[i, j]
                raw = cm[i, j]  # averaged count, may be fractional
                ax.text(j, i, f'{val*100:.1f}%\n({raw:.0f})',
                        ha='center', va='center', fontsize=9,
                        color='black')

        for i in range(3):
            ax.add_patch(plt.Rectangle(
                (i - 0.5, i - 0.5), 1, 1,
                fill=False, edgecolor='#c0392b', linewidth=2.5
            ))

        cbar = plt.colorbar(im, ax=ax, shrink=0.85, pad=0.04)
        cbar.set_label('Recall (row-normalised, %)', fontsize=8)
        cbar.ax.tick_params(labelsize=7.5)

        for i, (r, sup) in enumerate(zip(recall, support)):
            ax.text(3.9, i,
                    f'recall={r*100:.1f}%\nn={sup:.0f}',
                    ha='left', va='center',
                    fontsize=7.5, color='#444444',
                    transform=ax.transData)

        ax.set_xticks([0, 1, 2])
        ax.set_yticks([0, 1, 2])
        ax.set_xticklabels(BIN_LABELS, fontsize=10)
        ax.set_yticklabels(BIN_LABELS, fontsize=10)
        ax.set_xlabel('Predicted', fontsize=10)
        ax.set_ylabel('True', fontsize=10)
        ax.set_title(f'{ds_label} — {best_name}  (BA = {best_ba*100:.1f}%)',
                     fontsize=10, fontweight='bold', pad=10, color='#111111')
        ax.tick_params(length=0)

        for k in [0.5, 1.5]:
            ax.axhline(k, color='white', linewidth=1.5)
            ax.axvline(k, color='white', linewidth=1.5)

    plt.tight_layout()
    plt.show()


plot_confusion_matrices_best_models()

## 17. Feature importance (SHAP)

SHAP TreeExplainer computes exact Shapley values for tree models.
The best model per dataset is retrained on the full dataset (no CV)
to produce a single model for SHAP computation.

Because this is a 3-class problem, SHAP returns a separate set of
signed values per class. The top-20 ranking table uses the mean
ABSOLUTE SHAP value across all three classes (shap_abs below), which
gives an unbiased overall importance score. The beeswarm and net-
direction charts, however, show ONE class at a time (default: High
safety), since averaging signed values across classes causes them to
cancel out, particularly for weaker models where class contributions
are closer to symmetric.

Outputs:
- Beeswarm plot: top 20 individual features, direction + magnitude, one class
- Net direction plot: mean signed SHAP value per feature, one class
- Bar plot: top 10 feature families, mean absolute SHAP aggregated across classes
- Summary table: top 20 features with family label, ranked by cross-class mean |SHAP|

In [ ]:
## Standalone: get_best_clf (extracted from the ablation cell, without running ablation itself)
def get_best_clf(ds_label):
    best_name = max(results_all[ds_label], key=lambda r: r['bal_acc_mean'])['model']
    tp = tree_params[ds_label]
    if best_name == 'Deeper_MLP':
        return best_name, None
    elif best_name == 'XGBoost':
        return best_name, XGBClassifier(**tp['xgb'], subsample=0.8, colsample_bytree=0.8,
                                        eval_metric='mlogloss', random_state=SEED, verbosity=0)
    elif best_name == 'LightGBM':
        return best_name, LGBMClassifier(**tp['lgb'], subsample=0.8, colsample_bytree=0.8,
                                         random_state=SEED, verbosity=-1)
    elif best_name == 'CatBoost':
        return best_name, CatBoostClassifier(**tp['cat'], random_seed=SEED, verbose=0)


## 17. Feature importance (SHAP)
CLASS_NAMES = ('Low', 'Medium', 'High')

def get_family_shap(feat):
    if feat.startswith('comp_'):                                    return 'Composite'
    if feat.startswith('seg_') and feat.endswith('_cx'):           return 'Seg centroid X'
    if feat.startswith('seg_') and feat.endswith('_cy'):           return 'Seg centroid Y'
    if feat.startswith('seg_') and feat.endswith('_perimeter'):    return 'Seg perimeter'
    if feat.startswith('seg_') and re.search(r'_\d+_\d+$', feat): return 'Seg grid'
    if feat.startswith('seg_'):                                     return 'Seg coverage'
    if feat.startswith('obj_') and '_prob_'  in feat:              return 'Obj probability'
    if feat.startswith('obj_') and '_count_' in feat:              return 'Obj count'
    if feat.startswith('obj_') and '_area_'  in feat:              return 'Obj area'
    return 'Image statistics'

def retrain_best_model(ds_label):
    m      = merged_all[ds_label]
    feats  = feats_all[ds_label]
    X      = m[feats].reset_index(drop=True).fillna(0)
    y      = m['mu_bin'].reset_index(drop=True)
    sigma  = m[sigma_cols[ds_label]].reset_index(drop=True)
    cw_arr = compute_class_weight('balanced', classes=np.unique(y), y=y)
    cw     = {i: w for i, w in enumerate(cw_arr)}
    sw     = get_sample_weights(sigma.values, y.values, cw)
    best_name, clf_obj = get_best_clf(ds_label)
    clf = sklearn.base.clone(clf_obj)
    clf.fit(X, y, sample_weight=sw)
    families = [get_family_shap(f) for f in feats]
    print(f'  {ds_label}: retrained {best_name} on {len(X):,} samples, {len(feats)} features.')
    return clf, X, feats, families

print('=' * 70)
print('FEATURE IMPORTANCE — SHAP ANALYSIS')
print('=' * 70)
print('Best model per dataset retrained on full data for SHAP computation.')
print('TreeExplainer: exact Shapley values for tree models.')
print()

shap_results = {}
for ds_label in ['GPT', 'Human']:
    print(f'--- {ds_label} ---')
    clf, X_full, feat_names, families = retrain_best_model(ds_label)
    sample_size = min(2000, len(X_full))
    X_sample    = X_full.sample(n=sample_size, random_state=SEED)
    print(f'  Computing SHAP values on {sample_size:,} sampled images...')
    explainer   = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_sample)
    if isinstance(shap_values, list):
        shap_abs = np.mean([np.abs(sv) for sv in shap_values], axis=0)
    else:
        shap_abs = np.mean(np.abs(shap_values), axis=2) if shap_values.ndim == 3 else np.abs(shap_values)
    shap_results[ds_label] = {
        'clf': clf, 'explainer': explainer, 'shap_values': shap_values,
        'shap_abs': shap_abs, 'X_sample': X_sample,
        'feat_names': feat_names, 'families': families,
    }
    print(f'  SHAP values computed.')
    print()
print('SHAP computation complete.')


def _get_class_shap(res, target_class):
    """Returns the (n_samples, n_features) signed SHAP array for one class."""
    sv = res['shap_values']
    if isinstance(sv, list):
        return sv[target_class]
    elif sv.ndim == 3:
        return sv[:, :, target_class]
    else:
        return sv


## 17a. SHAP beeswarm — top 20 individual features (single class, fixed)
def plot_shap_beeswarm(ds_label, top_n=20, target_class=2):
    res        = shap_results[ds_label]
    X_sample   = res['X_sample']
    feat_names = res['feat_names']
    best_name  = get_best_clf(ds_label)[0]

    sv_class = _get_class_shap(res, target_class)
    mean_abs = np.abs(sv_class).mean(axis=0)
    top_idx  = np.argsort(mean_abs)[::-1][:top_n]
    top_feats = [feat_names[i] for i in top_idx]

    sv_top = sv_class[:, top_idx]
    X_top  = X_sample.iloc[:, top_idx].values

    fig, ax = plt.subplots(figsize=(12, 8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    for j in range(top_n - 1, -1, -1):
        feat_vals = X_top[:, j]
        sv_vals   = sv_top[:, j]
        f_min, f_max = feat_vals.min(), feat_vals.max()
        norm_vals = (feat_vals - f_min) / (f_max - f_min) if f_max > f_min else np.zeros_like(feat_vals)
        colors = plt.cm.RdBu_r(norm_vals)
        y_pos  = np.full(len(sv_vals), top_n - 1 - j)
        jitter = np.random.RandomState(SEED).uniform(-0.3, 0.3, len(sv_vals))
        ax.scatter(sv_vals, y_pos + jitter, c=colors, alpha=0.4, s=8, linewidths=0)

    ax.axvline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([f'{top_feats[top_n-1-i]}' for i in range(top_n)], fontsize=8)
    ax.set_xlabel(f'SHAP value for "{CLASS_NAMES[target_class]} safety" class', fontsize=10)
    ax.set_title(
        f'SHAP beeswarm — top {top_n} features\n'
        f'{ds_label} dataset | {best_name} | class: {CLASS_NAMES[target_class]} safety',
        fontsize=11, fontweight='bold'
    )
    sm   = plt.cm.ScalarMappable(cmap='RdBu_r', norm=plt.Normalize(0, 1))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.5, pad=0.02)
    cbar.set_label('Feature value (normalised)', fontsize=8)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(['Low', 'High'])
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=0.2, linestyle='--')
    plt.tight_layout()
    plt.show()
    plt.close(fig)

In [ ]:
## 17b. Net direction of feature effect — mean SIGNED SHAP value per feature
def plot_shap_net_direction(ds_label, top_n=20, target_class=2):
    res        = shap_results[ds_label]
    feat_names = res['feat_names']
    best_name  = get_best_clf(ds_label)[0]

    sv_class    = _get_class_shap(res, target_class)
    mean_signed = sv_class.mean(axis=0)
    top_idx     = np.argsort(np.abs(mean_signed))[::-1][:top_n]
    top_feats   = [feat_names[i] for i in top_idx]
    top_vals    = mean_signed[top_idx]

    order     = np.argsort(top_vals)
    top_feats = [top_feats[i] for i in order]
    top_vals  = top_vals[order]
    colors    = ['#c0392b' if v < 0 else '#1a7837' for v in top_vals]

    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')

    y_pos = np.arange(len(top_feats))
    ax.barh(y_pos, top_vals, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.8, linestyle='-', alpha=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(top_feats, fontsize=9)
    ax.set_xlabel(
        f'Mean SHAP value for "{CLASS_NAMES[target_class]} safety" class\n'
        f'(positive = pushes toward {CLASS_NAMES[target_class].lower()}, negative = pushes away)',
        fontsize=9
    )
    ax.set_title(
        f'Net direction of feature effect — top {top_n} by |mean SHAP|\n'
        f'{ds_label} dataset | {best_name} | class: {CLASS_NAMES[target_class]} safety',
        fontsize=11, fontweight='bold'
    )
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='x', alpha=0.2, linestyle='--')
    plt.tight_layout()
    plt.show()
    plt.close(fig)


print('=' * 70)
print('PLOT 1: SHAP BEESWARM — TOP 20 INDIVIDUAL FEATURES (HIGH SAFETY CLASS)')
print('=' * 70)
for ds_label in ['GPT', 'Human']:
    print(f'\n--- {ds_label} ---')
    plot_shap_beeswarm(ds_label, top_n=20, target_class=2)

print('=' * 70)
print('PLOT 2: NET DIRECTION OF FEATURE EFFECT (HIGH SAFETY CLASS)')
print('=' * 70)
for ds_label in ['GPT', 'Human']:
    print(f'\n--- {ds_label} ---')
    plot_shap_net_direction(ds_label, top_n=20, target_class=2)

In [ ]:
## 17c (alternative). Net direction of feature effect — as a DataFrame

def get_shap_direction_df(ds_label, top_n=20, target_class=2):
    res        = shap_results[ds_label]
    feat_names = res['feat_names']
    families   = res['families']
    best_name  = get_best_clf(ds_label)[0]

    sv_class    = _get_class_shap(res, target_class)
    mean_signed = sv_class.mean(axis=0)
    top_idx     = np.argsort(np.abs(mean_signed))[::-1][:top_n]

    rows = []
    for rank, idx in enumerate(top_idx, start=1):
        val = mean_signed[idx]
        rows.append({
            'rank': rank,
            'feature': feat_names[idx],
            'family': families[idx],
            'mean_shap': val,
            'direction': f'toward {CLASS_NAMES[target_class].lower()}' if val > 0 else f'away from {CLASS_NAMES[target_class].lower()}'
        })
    df = pd.DataFrame(rows)
    print(f'{ds_label} | {best_name} | class: {CLASS_NAMES[target_class]} safety')
    return df


gpt_direction_df   = get_shap_direction_df('GPT', top_n=20, target_class=2)
human_direction_df = get_shap_direction_df('Human', top_n=20, target_class=2)

display(gpt_direction_df)
display(human_direction_df)

In [ ]:
## 18. Spatial block cross-validation (London Plan sub-regions)
#
# Motivation: location-level GroupKFold (Section 13) prevents the same
# street point from appearing in both train and test but cannot prevent
# nearby points on the 50m grid from being in different folds. This section
# adds a stricter spatial validation using the five official London Plan
# sub-regions (Central, East, North, South, West) as fold boundaries.
# In each fold, one sub-region is held out as the test set while the model
# trains on the remaining four. The difference in BA between the two schemes
# quantifies geographical transferability.
#
# Reference: Greater London Authority, The London Plan (2021).
# Sub-regions: Central, East, North, South, West.
# Best model per dataset only (LightGBM for GPT, CatBoost for Human).

SUBREGIONS = ['Central', 'East', 'North', 'South', 'West']

def run_spatial_block_cv(ds_label):
    """
    Leave-one-sub-region-out cross-validation.
    Each of the 5 London Plan sub-regions is held out once as the test set.
    The best model per dataset is used with the same hyperparameters from
    the tuning step. Sigma-based sample weighting and balanced class weights
    are applied identically to the main CV loop.
    Returns: list of (subregion, n_train, n_test, ba) tuples.
    """
    m          = merged_all[ds_label]
    feats      = feats_all[ds_label]
    best_name, clf_template = get_best_clf(ds_label)
    sigma_col  = sigma_cols[ds_label]

    # Check sub-region coverage
    sr_counts = m['subregion'].value_counts()
    print(f'  Sub-region distribution in {ds_label} merged data:')
    for sr in SUBREGIONS:
        cnt = sr_counts.get(sr, 0)
        print(f'    {sr:<10}: {cnt:,} images')
    print()

    fold_results = []
    for sr_test in SUBREGIONS:
        train_mask = m['subregion'] != sr_test
        test_mask  = m['subregion'] == sr_test

        if test_mask.sum() == 0:
            print(f'  Sub-region {sr_test}: no test samples — skipped')
            continue
        if train_mask.sum() == 0:
            print(f'  Sub-region {sr_test}: no train samples — skipped')
            continue

        X_tr = m.loc[train_mask, feats].reset_index(drop=True)
        y_tr = m.loc[train_mask, 'mu_bin'].reset_index(drop=True)
        s_tr = m.loc[train_mask, sigma_col].reset_index(drop=True)
        X_te = m.loc[test_mask,  feats].reset_index(drop=True)
        y_te = m.loc[test_mask,  'mu_bin'].reset_index(drop=True)

        # Check class coverage in test set
        n_classes_te = y_te.nunique()
        if n_classes_te < NUM_BINS:
            print(f'  Sub-region {sr_test}: WARNING — only {n_classes_te} classes in test set')

        cw_arr = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
        cw     = {i: w for i, w in enumerate(cw_arr)}
        sw_tr  = get_sample_weights(s_tr.values, y_tr.values, cw)

        clf = sklearn.base.clone(clf_template)
        clf.fit(X_tr.fillna(0), y_tr, sample_weight=sw_tr)
        y_pred = clf.predict(X_te.fillna(0))
        ba     = balanced_accuracy_score(y_te, y_pred)

        fold_results.append({
            'subregion': sr_test,
            'n_train'  : len(X_tr),
            'n_test'   : len(X_te),
            'ba'       : ba
        })
        print(f'  Hold-out {sr_test:<10}: n_train={len(X_tr):,} | n_test={len(X_te):,} | BA={ba:.4f}')

    return fold_results


print('=' * 70)
print('SPATIAL BLOCK CROSS-VALIDATION — LONDON PLAN SUB-REGIONS')
print('=' * 70)
print('Scheme : leave-one-sub-region-out (5 folds)')
print('Groups : Central, East, North, South, West (GLA London Plan 2021)')
print('Model  : best model per dataset (same hyperparameters as main CV)')
print()

spatial_cv_results = {}

for ds_label in ['GPT', 'Human']:
    print(f'--- {ds_label} ---')
    best_name = max(results_all[ds_label], key=lambda r: r['bal_acc_mean'])['model']
    print(f'  Best model: {best_name}')
    fold_res = run_spatial_block_cv(ds_label)
    spatial_cv_results[ds_label] = fold_res
    bas  = [f['ba'] for f in fold_res]
    mean_ba = np.mean(bas)
    std_ba  = np.std(bas)
    print(f'\n  Mean BA across sub-regions: {mean_ba:.4f} +/- {std_ba:.4f}')
    print()

# Comparison table
print('=' * 70)
print('COMPARISON: LOCATION-GROUPED CV vs SPATIAL BLOCK CV')
print('=' * 70)
print(f'{"Dataset":<10} {"Model":<12} {"Location-grouped BA":>22} {"Spatial block BA":>20} {"Degradation":>14}')
print('-' * 82)
for ds_label in ['GPT', 'Human']:
    best_name  = max(results_all[ds_label], key=lambda r: r['bal_acc_mean'])['model']
    loc_ba     = lookups_all[ds_label][best_name]['bal_acc_mean']
    loc_std    = lookups_all[ds_label][best_name]['bal_acc_std']
    bas        = [f['ba'] for f in spatial_cv_results[ds_label]]
    sp_ba      = np.mean(bas)
    sp_std     = np.std(bas)
    degradation = loc_ba - sp_ba
    print(f'{ds_label:<10} {best_name:<12} '
          f'{loc_ba:.4f} +/- {loc_std:.4f}  '
          f'{sp_ba:.4f} +/- {sp_std:.4f}  '
          f'{degradation:>+.4f} pp')
print('-' * 82)
print()
print('Positive degradation = performance drops under stricter spatial separation.')
print('This quantifies the geographical transferability of the prediction model.')

# Per-sub-region bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('white')

for ax, ds_label in zip(axes, ['GPT', 'Human']):
    ax.set_facecolor('white')
    fold_res   = spatial_cv_results[ds_label]
    best_name  = max(results_all[ds_label], key=lambda r: r['bal_acc_mean'])['model']
    loc_ba     = lookups_all[ds_label][best_name]['bal_acc_mean']
    srs  = [f['subregion'] for f in fold_res]
    bas  = [f['ba']        for f in fold_res]
    color = DATASET_COLORS[ds_label]
    bars  = ax.bar(srs, bas, color=color, alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, bas):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9, color='#111111')
    ax.axhline(loc_ba, color='#c0392b', linestyle='--', linewidth=1.5,
               label=f'Location-grouped BA ({loc_ba*100:.1f}%)')
    ax.axhline(1/NUM_BINS, color='grey', linestyle=':', linewidth=1.2,
               label=f'Random baseline ({1/NUM_BINS*100:.1f}%)')
    ax.set_ylim(0, max(bas + [loc_ba]) * 1.15)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v*100:.0f}%'))
    ax.set_title(f'Spatial block CV — {ds_label} | {best_name}',
                 fontsize=11, fontweight='bold')
    ax.set_xlabel('Held-out sub-region (London Plan)', fontsize=9)
    ax.set_ylabel('Balanced accuracy', fontsize=9)
    ax.legend(fontsize=8, frameon=True, framealpha=0.9, edgecolor='#cccccc')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.2, linestyle='--')

plt.suptitle('Leave-one-sub-region-out CV: balanced accuracy per held-out sub-region\n'
             'Red dashed line = location-grouped CV baseline',
             fontsize=11, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()